# Analyzing Images with the Azure AI Vision SDK

> - **Door #1 — the dedicated Azure AI Vision service** (`azure-ai-vision-imageanalysis`): captions,
>   tags, objects with bounding boxes, and **Read/OCR** — each result carries a **confidence**.
>   Cheap, deterministic, built for scale.
> - **Door #2 — a multimodal LLM** (the vision-capable Foundry deployment): hand it an
>   image and a question and it *reasons* about the scene.
>
> This is the same **dedicated-service-vs-LLM** choice you make for *text* with Azure Language, now for *pixels*.

## Setup

We analyze a single **hosted** enterprise scene by URL — an office presentation photo, standing in for
a support-ticket attachment. Nothing to download for Steps 1–4.

Install the pinned SDKs (both doors):

In [ ]:
%pip install "azure-ai-vision-imageanalysis==1.0.0" "openai==1.54.4" "python-dotenv==1.0.1"

Import the Vision SDK. `VisualFeatures` is the menu of tasks (CAPTION, TAGS, OBJECTS, READ, ...) —
you get back only the features you ask for.

In [ ]:
import os

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures

load_dotenv()

### Credentials and the client factory

The demo reads the Vision **endpoint** and **key** from environment variables
(`AZURE_VISION_ENDPOINT` / `AZURE_VISION_KEY`) — copy both from the *same* resource's
*Keys and Endpoint* blade. The `os.getenv(..., default)` fallback lets this notebook run against the
shared training resource even before you've added the keys to your `.env`.

> **Warning:** never commit a real key. Hard-coding one here is fine for a throwaday classroom demo;
> in production it lives in an environment variable or Key Vault, and `.env` is git-ignored. A leaked
> key is the most common AI-project incident.

In [ ]:
# Endpoint + key — door #1. Prefer env vars; fall back to the shared training resource.
ENDPOINT = os.getenv("AZURE_VISION_ENDPOINT")
KEY = os.getenv("AZURE_VISION_KEY")

# A real hosted enterprise scene: an office presentation. Stand-in for a ticket attachment.
SAMPLE_IMAGE = ("https://learn.microsoft.com/azure/ai-services/"
                "computer-vision/media/quickstarts/presentation.png")


def client() -> ImageAnalysisClient:
    """Endpoint + key — the start of every Azure AI Vision call (the dedicated-service door)."""
    return ImageAnalysisClient(endpoint=ENDPOINT, credential=AzureKeyCredential(KEY))

## Step 1 — Classification & context: a caption and tags 

Run the **classification** and **contextual analysis** tasks in one call:

- a one-line **caption** (what is this scene?)
- a set of **tags** (what's in it?) — whole-image *classification*,

each with a **confidence**. You request *visual features* and get back only what you ask for, so a
single round trip can run several tasks at once

In [ ]:
# Ask for several visual features in ONE call — the service runs each and returns all.
result = client().analyze_from_url(
    image_url=SAMPLE_IMAGE,
    visual_features=[VisualFeatures.CAPTION, VisualFeatures.TAGS],
    gender_neutral_caption=True,   # "person" not "man"/"woman" — a Responsible-AI default
)

# CAPTION = contextual analysis: one sentence describing the whole scene, with a confidence.
if result.caption is not None:
    print(f"Caption: {result.caption.text!r}  (confidence {result.caption.confidence:.2f})")

# TAGS = whole-image classification: many labels, each with its own probability (the softmax again).
print("\nTop tags:")
for tag in result.tags.list[:6]:
    print(f"   {tag.name:16} {tag.confidence:.2f}")

**What you're looking at.** `gender_neutral_caption=True` is a Responsible-AI knob — it returns
"person" instead of guessing a gender (the fairness principle, surfaced as one parameter).
The caption confidence (~0.77) runs lower than the top tags (~1.00) because **captioning is the hardest
task** — a whole sentence is a bigger bet than a single label. `text 1.00` means the model is certain
the image contains text (there's a screen).

> **Region note.** Caption / dense-captions / people / smart-crops require a **caption-supported
> region** (East US, West US, West Europe, ...). Tags, Objects, and Read work everywhere. If CAPTION
> 400s, drop it and keep TAGS.

## Step 2 — Detection: objects with bounding boxes

Object **detection** is the difference between *"the image contains a person"* (classification) and
*"there's a person **here**, at these coordinates"* (detection). Same client, one different feature —
each found object comes back with a **bounding box** and its own **confidence**.

In [ ]:
# OBJECTS = detection: each found object gets a label AND a bounding box (x, y, w, h) in pixels.
det = client().analyze_from_url(
    image_url=SAMPLE_IMAGE,
    visual_features=[VisualFeatures.OBJECTS],
)

objects = det.objects.list if det.objects else []
print(f"Objects detected: {len(objects)}")
for obj in objects:
    box = obj.bounding_box               # an ImageBoundingBox: .x, .y, .width, .height
    label = obj.tags[0]                  # each object carries its own label + confidence
    print(f"   {label.name:14} {label.confidence:.2f}   "
          f"box=({box.x},{box.y}) {box.width}x{box.height}")

**What you're looking at.** `bounding_box` is `(x, y, width, height)` in pixels from the top-left
corner — what you'd draw a rectangle from to highlight the object, or crop to redact it. Each object's
`tags[0]` confidence is the *same softmax number*, now attached to a **region** instead of the whole
image; a low one (e.g. `laptop 0.62`) is the model telling you it's unsure about that box.

This is **named-entity recognition for images** — NER pulls `{PERSON, ORG, DATE}` spans
*with offsets* out of text; detection pulls `{person, chair, laptop}` regions *with boxes* out of an
image. Unstructured input → located, typed, confidence-scored entities.

## Step 3 — Read: text off the image

The **Read** (OCR) task lifts the *text* out of the image with per-word confidence. The result is
nested: **blocks → lines → words**, each word carrying its own `.confidence`. This is the capability
information extraction builds on.

In [ ]:
# READ = OCR: recognize printed/handwritten text, as blocks -> lines -> words.
ocr = client().analyze_from_url(
    image_url=SAMPLE_IMAGE,
    visual_features=[VisualFeatures.READ],
)

lines = []
if ocr.read is not None:
    for block in ocr.read.blocks:
        lines.extend(block.lines)        # each line has .text and a list of .words

print(f"Text lines read: {len(lines)}")
for line in lines[:5]:
    print(f"   {line.text!r}")

# Word-level confidence — the same per-token probability idea as STT.
words = [w for line in lines for w in line.words]
low = [w for w in words if w.confidence < 0.90]
print(f"\nWords read: {len(words)}   below 0.90 confidence: {len(low)}")
for w in low[:3]:
    print(f"   {w.text!r} @ {w.confidence:.2f}")

**The hand-off to information extraction — say it explicitly.** Read gives you a **flat blob of text and its
positions**. It does *not* tell you which line is the `total`, which is the `invoice_date`, which is the
`vendor`. Turning that blob into **named, typed fields** is **information extraction** —
Document Intelligence. **Vision reads; Document Intelligence understands.** Today you built the first half.

Notice the low-confidence words: long number codes and proper nouns sit lower because their glyphs are
ambiguous (`1` vs `l`, `0` vs `O`) — exactly what the softmax shape predicts.

## Step 4 — The other door: image-to-text with a multimodal model

Door #2 is **not** a separate vision SDK — it's the **multimodal LLM you already deployed**,
handed an image. We reuse the `openai` client and add one thing: an `input_image` part in the
message `content` list. Then we ask an open-ended question the dedicated service *can't* answer —
*what's going on in this attachment, and should we triage it?*

We load the Azure OpenAI values the same way the other notebooks in this folder do — from
`support-agents/.env`. The deployment **must be vision-capable** (gpt-4o / gpt-4o-mini / gpt-5 class).

In [ ]:
from openai import OpenAI

# The Azure OpenAI values live in support-agents/.env (same as the other notebooks here).
load_dotenv("../../support-agents/.env")


def _v1_base(url: str) -> str:
    """Normalize any data-plane endpoint form to the /openai/v1 base."""
    url = url.rstrip("/")
    for suffix in ("/responses", "/chat/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url if url.endswith("/openai/v1") else url + "/openai/v1"


def make_client() -> OpenAI:
    """The data-plane client — unchanged. A multimodal model is this model, fed an image."""
    return OpenAI(
        base_url=_v1_base(os.environ["AZURE_OPENAI_ENDPOINT"]),
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
    )


DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]   # must be a VISION-CAPABLE deployment
DEPLOYMENT

In [ ]:
# The multi-part content list: a text part AND an image part in ONE user message.
resp = make_client().responses.create(
    model=DEPLOYMENT,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text",
             "text": "This image is attached to a facilities support ticket. "
                     "In two sentences, describe what it shows and whether it needs triage."},
            {"type": "input_image", "image_url": SAMPLE_IMAGE},   # a URL or a data: base64 string
        ],
    }],
)

print(resp.output_text)

**Notice what this door does that door #1 can't.** The dedicated service returns *labels, boxes,
and text* — structured, but it doesn't **reason**. The LLM read the scene *and judged it* ("routine,
low priority, no triage"). That open-ended reasoning is the LLM's edge — and its cost: slower, pricier,
non-deterministic (run it twice, get different words). This is the dedicated-service-vs-LLM bake-off, now for images.

`image_url` takes a public URL *or* a base64 `data:` string (`f"data:image/png;base64,{b64}"`) for a
local file — one field, two ways to supply the pixels (see the stretch goals).

## Step 5 — Choosing a door: a confidence-gated router

Make the two-doors choice concrete *and responsible*: use the **cheap dedicated service** to caption and
classify at scale, **auto-accept** results whose confidence clears a bar, and **route the uncertain ones
to a human** (who might then spend the LLM call). This is the pattern every production vision pipeline is
built around — and you don't pick one door globally, you route **per item** on confidence.

In [ ]:
# The confidence bar for auto-accepting a caption/tag without human eyes.
# Threshold is a BUSINESS decision — tighter = safer but more manual review.
CONFIDENCE_THRESHOLD = 0.70


def triage(image_url: str) -> tuple[str, str, list[str]]:
    """Door #1 at scale: caption + tag an attachment, then decide auto-accept vs. human review.
    Returns (decision, caption_text, list_of_reasons)."""
    r = client().analyze_from_url(
        image_url=image_url,
        visual_features=[VisualFeatures.CAPTION, VisualFeatures.TAGS],
        gender_neutral_caption=True,
    )
    reasons: list[str] = []
    caption_text = r.caption.text if r.caption else "<no caption>"

    # A caption can fail two ways: absent (region/declined) OR present-but-uncertain. Block on both.
    if r.caption is None or r.caption.confidence < CONFIDENCE_THRESHOLD:
        conf = r.caption.confidence if r.caption else 0.0
        reasons.append(f"caption weak ({conf:.2f})")
    top = r.tags.list[0].confidence if r.tags and r.tags.list else 0.0
    if top < CONFIDENCE_THRESHOLD:
        reasons.append(f"top tag weak ({top:.2f})")

    decision = "AUTO-ACCEPT" if not reasons else "HUMAN REVIEW"
    return decision, caption_text, reasons


decision, caption_text, reasons = triage(SAMPLE_IMAGE)
print(f"Attachment triage: {decision}")
print(f"   caption: {caption_text!r}")
for reason in reasons:
    print(f"   flagged: {reason}")

**The threshold is a business decision — discuss it.** `0.70` is arbitrary. A safety-critical use
(*is the worker wearing a hard hat?*) might demand `0.95` and review everything else; a low-stakes
alt-text generator might auto-accept at `0.50`. You're trading throughput against risk — the same lever
as STT review and invoice auto-post.

> **Try it:** temporarily set `CONFIDENCE_THRESHOLD = 0.85` and re-run. The ~0.77 caption now flags and
> the decision flips to `HUMAN REVIEW` — not because the model got worse, but because *you* demanded more
> certainty. Restore it to `0.70`.